# Example: Training and Testing on a Sample Dataset

This notebook contains:

* Loading a sample of NDD dataset and preparing data for training.
* Preprocessing the dataset for training.
* Defining a simple CNN/INCC model for estimating the QC labels.
* Example code for main training loop.
* Example of how to forward CNN/ICNN features to RF classifier.
* Example of a single patient dataset evaluation.

In [38]:
import pandas as pd
import numpy as np

# for training loop
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Model definitions
import models as mdls

# for plotly visualizations 
from IPython.display import Javascript
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# render in notebook
from plotly.offline import init_notebook_mode
import plotly.io as pio
init_notebook_mode(connected=True)  
pio.renderers.default = 'iframe'

## Load Sample dataset

* Sample of NDD dataset with 100 voxels
* Run data preparation 

In [39]:
# Load a sample of 100 voxels from NDD dataset 

df_ndd = pd.read_pickle('ndd_sample_100voxels.pkl')
df_ndd

,Subject ID,Subject Visit,Subject Dx,spectrum,imspectrum,label
0,pt27,vt1,ms,"[22455802.0, 22455802.0, 22498086.0, 22599410....","[-34383170.0, -34383170.0, -34542750.0, -34803...",0.0
1,pt1,vt1,healthy,"[-48129692.0, -39246640.0, -29046868.0, -19700...","[322269500.0, 322483620.0, 320508300.0, 316000...",1.0
2,pt7,vt1,healthy,"[-780765950.0, -837825100.0, -880047500.0, -91...","[-1848889000.0, -1761272200.0, -1675768700.0, ...",0.0
3,pt30,vt1,ms,"[36944572.0, 37206428.0, 37357756.0, 37380252....","[8033980.5, 7706555.5, 7293155.5, 6836170.0, 6...",1.0
4,pt30,vt1,ms,"[3147212.8, 2783149.8, 2608026.8, 2605612.8, 2...","[-6280886.5, -5189969.5, -4141626.8, -3159236....",1.0
...,...,...,...,...,...,...
95,pt26,vt1,ms,"[211633310.0, 211633310.0, 211633310.0, 211633...","[408330720.0, 408330720.0, 408330720.0, 408330...",1.0
96,pt12,vt2,mdd,"[-313938900.0, -274703940.0, -239341620.0, -21...","[1555870600.0, 1515760500.0, 1474414500.0, 143...",1.0
97,pt8,vt1,healthy,"[245113180.0, 239170260.0, 233489890.0, 229471...","[-91366670.0, -91090420.0, -89579180.0, -86934...",0.0
98,pt9,vt1,healthy,"[329744100.0, 319173150.0, 310957220.0, 304973...","[-174205600.0, -173042940.0, -169855860.0, -16...",0.0


## Prepare Data for Training 
* Crop spectra and split in train/test
* Apply Standard Scalar and add cropped and scaled spectra to dataframe (`processed_spectrum`)

In [40]:
# crop spectra to metabolite region
X = np.array(df_ndd["spectrum"].tolist())
X = X[:, 1196:2046]

# labels 0 = good, 1 = bad --> one-hot encode
y = np.array(df_ndd["label"]).reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

#  train val split 
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# standardize the data using training statns 
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# add processed spectra to dataframe 
X_all_processed = scaler.transform(X)
df_ndd["processed_spectrum"] = list(X_all_processed)

# unsqueeze for channel dimensions 
X_train = np.expand_dims(X_train, axis=2)
X_val = np.expand_dims(X_val, axis=2)

# check dataset sizes 
print(f"train || validation size: {len(X_train)}, {len(X_val)}")
print(f"X_train: \t\t {X_train.shape}")
print(f"X_val: \t\t\t {X_val.shape}")
print(f"y_train: \t\t {y_train.shape}")
print(f"y_val: \t\t\t {y_val.shape}")

train || validation size: 90, 10
X_train: 		 (90, 850, 1)
X_val: 			 (10, 850, 1)
y_train: 		 (90, 2)
y_val: 			 (10, 2)


### Create Dataset Loaders for Torch Training 

In [41]:
# Convert to tensors
x_train = torch.tensor(X_train, dtype=torch.float32)
x_val = torch.tensor(X_val, dtype=torch.float32)

# Rearrange to (N, 1, 850) for Conv1d
x_train = x_train.transpose(1, 2).contiguous()
x_val = x_val.transpose(1, 2).contiguous()

y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

## Model initailization

In [42]:
# Initialize model as defined in models.py file
# Use ICNN_QC for inception model
model = mdls.CNN_QC()

In [43]:
model

CNN_QC(
  (conv1): Conv1d(1, 64, kernel_size=(5,), stride=(1,), padding=same)
  (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=same)
  (conv3): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=same)
  (conv4): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=same)
  (conv5): Conv1d(128, 64, kernel_size=(5,), stride=(1,), padding=same)
  (conv6): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=same)
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (dense1): Linear(in_features=832, out_features=64, bias=True)
  (dense2): Linear(in_features=64, out_features=64, bias=True)
  (dense3): Linear(in_features=64, out_features=2, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
)

## Model Training

### Example of Main Training Loop
* Set up optimizer
* Loss criterion
* Store accuracy metrics
* Train for 15 epochs

In [7]:

# Training setup
# Optimizer with L2 regularization for dense layers only (lambda = 0.01)
optimizer = Adam([
    {'params': [param for name, param in model.named_parameters() if 'dense' in name], 
     'weight_decay': 0.01},
    {'params': [param for name, param in model.named_parameters() if 'dense' not in name]}], lr=1e-4)


# use categorical cross-entropy
criterion = nn.CrossEntropyLoss()

# store trianing metrics
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

num_epochs = 15

for epoch in range(num_epochs):
    running_loss = 0
    correct_train = 0
    total_train = 0
    
    model.train()

    for idx, (x_batch, y_batch) in enumerate(train_loader): 
        
        optimizer.zero_grad()
        
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
    
        running_loss += loss.item()

        # Calculate training accuracy
        _, y_pred_train = torch.max(outputs, 1)
        _, y_true_train = torch.max(y_batch, 1)
        total_train += y_batch.size(0)
        correct_train += (y_pred_train == y_true_train).sum().item()

        # print statistics every 10th batch --> only if debuggging
        if 0 and idx % 10 == 0:    
            print(f'[{epoch + 1}, {idx + 1:5d}] loss: {running_loss / (idx + 1):.3f}')
    
    # average training loss and accuracy
    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct_train / total_train

    # store training loss and accuracy
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    
    # validate
    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            outputs = model(x_batch)
            val_loss += criterion(outputs, y_batch).item()
            _, y_pred_val = torch.max(outputs, 1)
            _, y_true_val = torch.max(y_batch, 1)
            total_val += y_batch.size(0)
            correct_val += (y_pred_val == y_true_val).sum().item()

    # Calculate average validation loss and accuracy
    val_loss /= len(val_loader)
    val_accuracy = 100 * correct_val / total_val
    
    # Append validation loss and accuracy
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Acc: {train_accuracy:.2f}%, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')

print('Finished Training')

Epoch 1/15, Loss: 0.6926, Acc: 50.00%, Val Loss: 0.6933, Val Accuracy: 50.00%
Epoch 2/15, Loss: 0.6926, Acc: 48.89%, Val Loss: 0.6932, Val Accuracy: 50.00%
Epoch 3/15, Loss: 0.6920, Acc: 53.33%, Val Loss: 0.6932, Val Accuracy: 50.00%
Epoch 4/15, Loss: 0.6917, Acc: 50.00%, Val Loss: 0.6931, Val Accuracy: 50.00%
Epoch 5/15, Loss: 0.6930, Acc: 50.00%, Val Loss: 0.6930, Val Accuracy: 50.00%
Epoch 6/15, Loss: 0.6944, Acc: 48.89%, Val Loss: 0.6929, Val Accuracy: 50.00%
Epoch 7/15, Loss: 0.6912, Acc: 51.11%, Val Loss: 0.6928, Val Accuracy: 50.00%
Epoch 8/15, Loss: 0.6923, Acc: 52.22%, Val Loss: 0.6926, Val Accuracy: 50.00%
Epoch 9/15, Loss: 0.6895, Acc: 53.33%, Val Loss: 0.6923, Val Accuracy: 50.00%
Epoch 10/15, Loss: 0.6924, Acc: 50.00%, Val Loss: 0.6921, Val Accuracy: 50.00%
Epoch 11/15, Loss: 0.6906, Acc: 52.22%, Val Loss: 0.6918, Val Accuracy: 50.00%
Epoch 12/15, Loss: 0.6925, Acc: 52.22%, Val Loss: 0.6914, Val Accuracy: 50.00%
Epoch 13/15, Loss: 0.6921, Acc: 50.00%, Val Loss: 0.6909, Val

#### Save the model checkpoint and preprocessing StandardScalar 

* Standard Scalar values should be stored to ensure appropriate pre-processing when making predictions

In [8]:
#### save model weights
torch.save(model.state_dict(), 'model_sample_params.ckpt')

# save preprocessing objects
import joblib
preprocessing = {"scaler": scaler, 
                 "encoder": encoder}
joblib.dump(preprocessing, "preprocessing_sample_params.pkl")

print("Saved model_sample_params.ckpt and preprocessing_sample_params.pkl")

Saved model_sample_params.ckpt and preprocessing_sample_params.pkl


### View Training Curves

In [9]:
fig = make_subplots(rows=1,cols=2,subplot_titles=("Training and Validation Loss", "Training and Validation Accuracy"))

# Train and val losses
fig.add_trace(go.Scatter(
        x=list(range(1, num_epochs + 1)), y=train_losses,
        mode="lines+markers",
        name="Training loss"
    ), row=1, col=1)

fig.add_trace( go.Scatter(x=list(range(1, num_epochs + 1)), y=val_losses,
        mode="lines+markers",
        name="Validation loss"
    ), row=1, col=1)

# Train and val accuracies
fig.add_trace(go.Scatter(
        x=list(range(1, num_epochs + 1)), y=train_accuracies,
        mode="lines+markers",
        name="Training accuracy"
    ), row=1, col=2)

fig.add_trace( go.Scatter(x=list(range(1, num_epochs + 1)), y=val_accuracies,
        mode="lines+markers",
        name="Validation accuracy"
    ), row=1, col=2)

fig.update_layout(
    width=1200,
    height=450,
    title="Model Training History",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(title_text="Epochs", showgrid=True, gridcolor="lightgray", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="lightgray", zeroline=False)

fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2)

fig.show()

### Optional RF training
* This code block shows an example of how to use the `return_features`input from the model to
* extract the last layer as features for input to a RF classifier
* Feature extraction is following main CNN/ICNN training
* Model is put in eval mode and evaluated for train and val datasets
* Once features are extracted, train (validate) data features are used to fit (evaluate) the RF classifier

#### Feature extraction 

In [10]:
# Put model back in eval mode and extract features at end of the dense layer 
model.eval()

# Training data - extract learned features from CNN/ICNN and combine batches
X_train_features = []
y_train_rf = []

with torch.no_grad():
    for x_batch, y_batch in train_loader:

        # Get output from dense2 (64 features)
        features = model(x_batch, return_features=True)

        X_train_features.append(features.cpu())
        y_train_rf.append(y_batch.cpu())

X_train_features = torch.cat(X_train_features).numpy()
y_train_rf = torch.cat(y_train_rf).numpy()

# Convert one-hot labels to class labels
y_train_rf = torch.argmax(torch.tensor(y_train_rf), dim=1).numpy()

print("Training CNN features:", X_train_features.shape)
print("Training RF labels:", y_train_rf.shape)


# Repeat for validation data - extract learned features from CNN/ICNN
X_val_features = []
y_val_rf = []

with torch.no_grad():
    for x_batch, y_batch in val_loader:

        # Get output from dense2 (64 features)
        features = model(x_batch, return_features=True)

        X_val_features.append(features.cpu())
        y_val_rf.append(y_batch.cpu())

X_val_features = torch.cat(X_val_features).numpy()
y_val_rf = torch.cat(y_val_rf).numpy()
y_val_rf = torch.argmax(torch.tensor(y_val_rf), dim=1).numpy()

Training CNN features: (90, 64)
Training RF labels: (90,)


####  Random Forest Fitting

In [11]:
# initailize the RF head
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, random_state=42)

# fit RF
rf.fit(X_train_features, y_train_rf)

print("Random Forest training complete")

# Predict on validation set 
y_pred_rf = rf.predict(X_val_features)

rf_accuracy = (y_pred_rf == y_val_rf).mean() * 100

print(f"Random Forest validation accuracy: {rf_accuracy:.2f}%")

Random Forest training complete
Random Forest validation accuracy: 70.00%


## Example: Test CNN on sample dataset

### Load model and preprocess vars

In [45]:
# Load preprocessing info
preprocessing = joblib.load("preprocessing_cnn.pkl")

# StandardScalar and one-hot encoder to ensure preprocessing matches training
scaler = preprocessing["scaler"]
encoder = preprocessing["encoder"]

# Load model 
model = mdls.CNN_QC()
model.load_state_dict(torch.load("model_cnn.ckpt", map_location="cpu", weights_only=True))
model.eval()

CNN_QC(
  (conv1): Conv1d(1, 64, kernel_size=(5,), stride=(1,), padding=same)
  (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=same)
  (conv3): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=same)
  (conv4): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=same)
  (conv5): Conv1d(128, 64, kernel_size=(5,), stride=(1,), padding=same)
  (conv6): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=same)
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (dense1): Linear(in_features=832, out_features=64, bias=True)
  (dense2): Linear(in_features=64, out_features=64, bias=True)
  (dense3): Linear(in_features=64, out_features=2, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
)

### Load data and preprocess 

In [46]:
# Set up the data for testing
df_pt = pd.read_pickle("example_healthy_pt.pkl")
df_pt

,row,column,slice,spectrum,imspectrum,label
0,4,7,2,"[691851260.0, 691851260.0, 693492100.0, 695640...","[596314560.0, 596314560.0, 571778300.0, 546021...",0.0
1,4,8,2,"[-559411800.0, -559411800.0, -546197500.0, -53...","[-96587620.0, -96587620.0, -76732696.0, -60055...",0.0
2,4,9,2,"[168567410.0, 169957180.0, 172895340.0, 175155...","[50881684.0, 52729932.0, 51597892.0, 49525776....",1.0
3,5,7,2,"[-7266171.0, -7266171.0, -22561730.0, -4162521...","[-431248160.0, -431248160.0, -440645800.0, -44...",1.0
4,5,8,2,"[-73047490.0, -73047490.0, -63118830.0, -52828...","[-367958700.0, -367958700.0, -364117060.0, -36...",0.0
...,...,...,...,...,...,...
387,17,7,5,"[-515080800.0, -515080800.0, -492550100.0, -47...","[498961730.0, 498961730.0, 490312160.0, 485949...",1.0
388,17,8,5,"[-974097500.0, -974097500.0, -966774900.0, -95...","[-372511230.0, -372511230.0, -339586430.0, -30...",1.0
389,17,9,5,"[-550137900.0, -541898700.0, -533466100.0, -51...","[-181333150.0, -149721330.0, -112386840.0, -71...",0.0
390,17,10,5,"[66539256.0, 43447284.0, 25023038.0, 10265052....","[948456900.0, 951649700.0, 957028160.0, 963865...",0.0


In [50]:
# crop spectra to metabolite region
X_pt = np.array(df_pt["spectrum"].tolist())
X_pt = X_pt[:, 1196:2046]

# Apply the same training scaler
X_pt = scaler.transform(X_pt)

# Save processed spectra
df_pt["processed_spectrum"] = list(X_pt)

# Convert to torch tensor
x_pt_test = torch.tensor(X_pt, dtype=torch.float32)

# unsqueeze for conv 1d
x_pt_test = x_pt_test.unsqueeze(1)

# true labels 0 = good, 1 = bad --> one hot encode 
y_pt = np.array(df_pt["label"]).reshape(-1, 1)

# Use the same  encoder from training and convert to torch tensor
y_pt = encoder.transform(y_pt)
y_pt = torch.tensor(y_pt, dtype=torch.float32 )

### Predict labels

In [51]:
# evaluate and take softmax for final prediction 
with torch.no_grad():
    outputs = model(x_pt_test)
    probs = torch.softmax(outputs, dim=1)
    df_pt["label_pred"] = torch.argmax(probs, dim=1)

In [52]:
df_pt

,row,column,slice,spectrum,imspectrum,label,processed_spectrum,label_pred
0,4,7,2,"[691851260.0, 691851260.0, 693492100.0, 695640...","[596314560.0, 596314560.0, 571778300.0, 546021...",0.0,"[-0.06784891, -0.07054317, -0.0737233, -0.0775...",0
1,4,8,2,"[-559411800.0, -559411800.0, -546197500.0, -53...","[-96587620.0, -96587620.0, -76732696.0, -60055...",0.0,"[-0.0059133717, 0.00043488934, 0.006728045, 0....",0
2,4,9,2,"[168567410.0, 169957180.0, 172895340.0, 175155...","[50881684.0, 52729932.0, 51597892.0, 49525776....",1.0,"[0.011560571, 0.007590902, 0.0064665857, 0.008...",1
3,5,7,2,"[-7266171.0, -7266171.0, -22561730.0, -4162521...","[-431248160.0, -431248160.0, -440645800.0, -44...",1.0,"[0.124653615, 0.13619019, 0.14508305, 0.148722...",0
4,5,8,2,"[-73047490.0, -73047490.0, -63118830.0, -52828...","[-367958700.0, -367958700.0, -364117060.0, -36...",0.0,"[0.094334036, 0.09296821, 0.087999284, 0.08319...",0
...,...,...,...,...,...,...,...,...
387,17,7,5,"[-515080800.0, -515080800.0, -492550100.0, -47...","[498961730.0, 498961730.0, 490312160.0, 485949...",1.0,"[-0.93507427, -0.902634, -0.88001883, -0.86602...",1
388,17,8,5,"[-974097500.0, -974097500.0, -966774900.0, -95...","[-372511230.0, -372511230.0, -339586430.0, -30...",1.0,"[-0.9028731, -0.87854654, -0.8498795, -0.82246...",1
389,17,9,5,"[-550137900.0, -541898700.0, -533466100.0, -51...","[-181333150.0, -149721330.0, -112386840.0, -71...",0.0,"[-0.55513227, -0.5540059, -0.5464505, -0.53025...",1
390,17,10,5,"[66539256.0, 43447284.0, 25023038.0, 10265052....","[948456900.0, 951649700.0, 957028160.0, 963865...",0.0,"[-0.5377322, -0.5296645, -0.521543, -0.5099070...",0


#### View predicted labels

##### Create label masks for visualization and overlap
* Red = predicted bad
* Blue = true label bad
* Purple = overlaps (correctly labeled --> will display as purple)
* So red/blue are those that are either false negatives (red) or false positives (blue)
* no color = both predicted and true labels are "good" 

In [53]:
# data volume is 18 x 22 x 8 slices

# create label masks 
true_label_mask = np.zeros((18, 22, 8), dtype=np.int32)
pred_label_mask = np.zeros((18, 22, 8), dtype=np.int32)

# Fill masks
for _, r in df_pt.iterrows():
    i = int(r["row"])
    j = int(r["column"])
    k = int(r["slice"])

    true_label_mask[i, j, k] = r["label"]
    pred_label_mask[i, j, k] = r["label_pred"]

In [54]:
# view slices that are within the brain region
slices = [2, 3, 4, 5]

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f"Slice {s}" for s in slices],
    horizontal_spacing=0.03
)

for i, s in enumerate(slices):
    # True label
    fig.add_trace(go.Heatmap(z=true_label_mask[:, :, s],
            colorscale=[[0, "rgba(255,255,255,0)"], [1, "blue"]],
            zmin=0, zmax=1, opacity=0.8, showscale=False,
            name="True label", legendgroup="True label",
            showlegend=(i == 0)
        ), row=1, col=i+1)

    # Prediction
    fig.add_trace(
        go.Heatmap(
            z=pred_label_mask[:, :, s],
            colorscale=[[0, "rgba(255,255,255,0)"], [1, "red"]],
            zmin=0, zmax=1, opacity=0.5, showscale=False,
            name="Prediction", legendgroup="Prediction",
            showlegend=(i == 0)
        ), row=1, col=i+1 )

fig.update_layout(
    title="True (Blue) vs Prediction (Red) Label Masks",
    width=800, height=250,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=10, r=10, t=50, b=50),
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center", yanchor="top" ))

fig.update_xaxes(showgrid=False, showticklabels=False, zeroline=False)

fig.update_yaxes(
    showgrid=False,
    showticklabels=False,
    zeroline=False,
    scaleanchor="x"
)

fig.show()

#### Example correctly labeled voxels

* 0 = good
* 1 = bad

In [55]:
# View a few voxels that were correctly labeled 
idx_match = np.where(
    ((df_pt["label"] == 1) & (df_pt["label_pred"] == 1)) |
    ((df_pt["label"] == 0) & (df_pt["label_pred"] == 0))
)[0]

rng = np.random.default_rng()
sample_idx = rng.choice(idx_match, size=12, replace=False)

# Make a subplot 
fig = make_subplots(
    rows=3,
    cols=4,
    subplot_titles=[
        f"Voxel ({df_pt.iloc[idx]['row']}, {df_pt.iloc[idx]['column']}, {df_pt.iloc[idx]['slice']})<br>"
        f"True={df_pt.iloc[idx]['label']}, Pred={df_pt.iloc[idx]['label_pred']}"
        for idx in sample_idx
    ]
)

for i, idx in enumerate(sample_idx):
    row_idx = i // 4 + 1
    col_idx = i % 4 + 1

    row = df_pt.iloc[idx]

    fig.add_trace(
        go.Scatter(
            y=np.asarray(row["processed_spectrum"]),
            mode="lines",
            showlegend=False
        ),
        row=row_idx,
        col=col_idx
    )

fig.update_layout(
    width=1200,
    height=800,
    title=" Correctly Labeled Spectra",
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="closest"
)

fig.update_xaxes(
    title_text="Spectrum index",
    showgrid=True,
    gridcolor="lightgray",
    zeroline=False
)

fig.update_yaxes(
    title_text="A.U.",
    showgrid=True,
    gridcolor="lightgray",
    zeroline=False
)

fig.show()

#### Example mislabeled voxels

* 0 = good
* 1 = bad

In [56]:
# View a few voxels that were mislabeled 
idx_mismatch = np.where(
    ((df_pt["label"] == 0) & (df_pt["label_pred"] == 1)) |
    ((df_pt["label"] == 1) & (df_pt["label_pred"] == 0))
)[0]

rng = np.random.default_rng(42)
sample_idx = rng.choice(idx_mismatch, size=12, replace=False)

# Make a subplot 
fig = make_subplots(
    rows=3,
    cols=4,
    subplot_titles=[
        f"Voxel ({df_pt.iloc[idx]['row']}, {df_pt.iloc[idx]['column']}, {df_pt.iloc[idx]['slice']})<br>"
        f"True={df_pt.iloc[idx]['label']}, Pred={df_pt.iloc[idx]['label_pred']}"
        for idx in sample_idx
    ]
)

for i, idx in enumerate(sample_idx):
    row_idx = i // 4 + 1
    col_idx = i % 4 + 1

    row = df_pt.iloc[idx]

    fig.add_trace(
        go.Scatter(
            y=np.asarray(row["processed_spectrum"]),
            mode="lines",
            showlegend=False
        ),
        row=row_idx,
        col=col_idx
    )

fig.update_layout(
    width=1200,
    height=800,
    title=" Misclassified Spectra",
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="closest"
)

fig.update_xaxes(
    title_text="Spectrum index",
    showgrid=True,
    gridcolor="lightgray",
    zeroline=False
)

fig.update_yaxes(
    title_text="A.U.", showgrid=True,
    gridcolor="lightgray",
    zeroline=False
)

fig.show()